# DEV — FastAPI Serving Endpoint

**Duration:** 5 min

## Build the API

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib, numpy as np, pandas as pd

app = FastAPI(title='Churn Prediction API', version='1.0')
bundle = joblib.load('model.pkl')

class CustomerFeatures(BaseModel):
    tenure: int
    MonthlyCharges: float
    TotalCharges: float
    Contract: int          # 0=Month-to-month, 1=One year, 2=Two year
    InternetService: int   # 0=DSL, 1=Fiber optic, 2=No
    # ... add remaining features

@app.get('/health')
def health():
    return {'status': 'ok', 'model_features': len(bundle['features'])}

@app.post('/predict')
def predict(customer: CustomerFeatures):
    try:
        X = pd.DataFrame([customer.dict()])[bundle['features']]
        prob = bundle['model'].predict_proba(X)[0, 1]
        return {
            'churn_probability': round(float(prob), 4),
            'churn_predicted': bool(prob >= 0.5),
            'risk': 'high' if prob > 0.7 else 'medium' if prob > 0.4 else 'low'
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/mlops-real-world/mlops-dev-api.ipynb)


## Test Locally

In [ ]:
uvicorn api.app:app --reload
# Open http://localhost:8000/docs for Swagger UI